# Stream IMDb Reviews to the Bronze Layer (MinIO)

This tutorial walks through streaming the IMDb Movie Reviews dataset into a **bronze layer** in MinIO (S3-compatible object storage).

**Data flow:**

1. **Load** IMDb reviews from Hugging Face `datasets`
2. **Produce** them to a Kafka topic (`imdb-reviews`)
3. **Consume** from Kafka and write JSONL files to MinIO under `bronze/imdb/`

**Prerequisites:**

- Docker Desktop running
- Kafka and MinIO started: `docker compose -f docker/docker-compose.yml up -d kafka minio`
- Python packages: `pip install -r requirements.txt` (or run the install cell below)

## 1. Start Kafka and MinIO

From a terminal in the project root:

```bash
docker compose -f docker/docker-compose.yml up -d kafka minio
```

Wait until Kafka logs show `Kafka Server started`. MinIO is ready when the S3 API responds at http://localhost:9000.

In [ ]:
# Install dependencies (run once)
#!pip install -q kafka-python datasets boto3

## 2. Produce IMDb reviews to Kafka

## Schema: `ImdbBronzeReview`

All IMDb reviews in this pipeline use the **`ImdbBronzeReview`** schema (from `src.utils.schema`). It defines a normalized record shape:

| Field  | Type | Description                    |
|--------|------|--------------------------------|
| `id`   | str  | Unique identifier (e.g. row index) |
| `text` | str  | Raw review text                |
| `label`| 0 or 1 | Binary sentiment: 0 = negative, 1 = positive |

**What was done:**
- **Producer**: Raw IMDb rows (from Hugging Face) are normalized with `ImdbBronzeReview.from_raw_imdb(row, idx)`, then serialized via `to_message()` and sent to Kafka.
- **Consumer**: JSON from Kafka is validated into `ImdbBronzeReview(id, text, label)` and written to bronze as JSONL. Same schema end-to-end keeps the pipeline consistent.

**Schema definition** (`src/utils/schema.py`):

```python
from dataclasses import dataclass
from typing import Literal

LabelType = Literal[0, 1]

@dataclass
class ImdbBronzeReview:
    id: str
    text: str
    label: LabelType  # 0 or 1

    @classmethod
    def from_raw_imdb(cls, raw: dict, idx: int) -> "ImdbBronzeReview":
        """Create from raw IMDb dataset row."""
        text = str(raw.get("text", ""))
        label_raw = raw.get("label")
        if label_raw not in (0, 1):
            raise ValueError(f"Unexpected label: {label_raw!r}")
        return cls(id=str(idx), text=text, label=label_raw)

    def to_message(self) -> dict:
        """Serialize for Kafka JSON."""
        return {"id": self.id, "text": self.text, "label": int(self.label)}
```

In [ ]:
# Example: create from raw IMDb row and serialize for Kafka
from src.utils.schema import ImdbBronzeReview

raw = {"text": "Great film!", "label": 1}
review = ImdbBronzeReview.from_raw_imdb(raw, idx=0)
print("ImdbBronzeReview:", review)
print("Kafka message:", review.to_message())

### 2.1 Imports and configuration

We use `src.config` for Kafka and topic settings, and `ImdbBronzeReview` to normalize each record.

In [ ]:
# Imports
import json
import time
from datasets import load_dataset
from kafka import KafkaProducer
from kafka.errors import NoBrokersAvailable

from src import config
from src.utils.schema import ImdbBronzeReview

In [ ]:
# Config
bootstrap = config.KAFKA_BOOTSTRAP_SERVERS
topic = config.KAFKA_IMDB_TOPIC
limit = 24999

### 2.2 Connect to Kafka

Retry a few times in case the broker is still starting.

### 2.3 Run the producer

Connect to Kafka, load the IMDb train split, normalize each record, and send to the topic.

In [ ]:
# Connect to Kafka (retry if broker still starting)
for attempt in range(5):
    try:
        producer = KafkaProducer(
            bootstrap_servers=bootstrap,
            value_serializer=lambda v: json.dumps(v).encode("utf-8"),
        )
        break
    except NoBrokersAvailable:
        print("Kafka not ready, retrying...")
        time.sleep(2)
else:
    raise RuntimeError("Kafka not reachable. Is it running?")

In [ ]:
# Load IMDb dataset (downloads on first run) and stream to Kafka
ds = load_dataset("imdb", split="train")
count = 0
for idx, row in enumerate(ds):
    if idx >= limit:
        break
    review = ImdbBronzeReview.from_raw_imdb(row, idx, split="train")
    producer.send(topic, review.to_message())
    count += 1

producer.flush()
producer.close()
print(f"Produced {count} reviews to topic '{topic}'.")

### 3.0 Imports and setup

Import consumer dependencies and create the Kafka consumer.

## 3. Consume from Kafka and write to MinIO (bronze layer)

We consume messages from the `imdb-reviews` topic, buffer them in memory, and upload batches as **JSONL** (newline-delimited JSON) files to MinIO under `bronze/imdb/`. Each file gets a unique key with timestamp and UUID.

In [ ]:
# Imports and config
import json
import time
from uuid import uuid4
from kafka import KafkaConsumer

from src import config
from src.utils.schema import ImdbBronzeReview
from src.utils.s3_client import ensure_bucket_exists, upload_bytes

bootstrap = config.KAFKA_BOOTSTRAP_SERVERS
topic = config.KAFKA_IMDB_TOPIC
batch_size = 100
max_messages = 25000

ensure_bucket_exists()
# Use a unique group_id so we read from earliest when running a batch demo
consumer = KafkaConsumer(
    topic,
    bootstrap_servers=bootstrap,
    auto_offset_reset="earliest",
    group_id=f"bronze-tutorial-{int(time.time())}",
    value_deserializer=lambda x: json.loads(x.decode("utf-8")),
    consumer_timeout_ms=10000,
)

### 3.1 Consume loop

Consume messages, validate, buffer, and upload batches to MinIO.

In [ ]:
# Consume messages, buffer, and flush batches to MinIO
buffer = []
total = 0
record_split = "train"  # split from payload, default train
for msg in consumer:
    payload = msg.value
    try:
        record_split = str(payload.get("split", "train"))
        record = ImdbBronzeReview(
            id=str(payload.get("id")),
            text=str(payload.get("text", "")),
            label=int(payload.get("label")),
            split=record_split,
        )
    except Exception as e:
        print(f"Skip invalid: {e}")
        continue

    buffer.append(json.dumps(record.to_message(), ensure_ascii=False))
    total += 1

    if len(buffer) >= batch_size:
        key = f"{config.BRONZE_PREFIX}imdb/{record_split}/imdb_bronze_{int(time.time())}_{uuid4().hex}.jsonl"
        data = ("\n".join(buffer) + "\n").encode("utf-8")
        upload_bytes(key, data)
        print(f"Wrote {len(buffer)} records to s3://{config.S3_DATA_BUCKET}/{key}")
        buffer.clear()

    if total >= max_messages:
        break

if buffer:
    key = f"{config.BRONZE_PREFIX}imdb/{record_split}/imdb_bronze_{int(time.time())}_{uuid4().hex}.jsonl"
    data = ("\n".join(buffer) + "\n").encode("utf-8")
    upload_bytes(key, data)
    print(f"Wrote {len(buffer)} records to s3://{config.S3_DATA_BUCKET}/{key}")

consumer.close()
print(f"Consumed {total} reviews and wrote to bronze layer.")

---

## Summary

You have streamed IMDb reviews: **Dataset → Kafka → Bronze (MinIO)**. Downstream jobs (e.g. `silver_job.py`, `embedding_job.py`) can read from `bronze/imdb/` for cleaning and feature engineering.

### 4.1 Sample records

Read the first JSONL file and print a couple of records.

## 4. Verify bronze objects in MinIO

Use the S3 client to list objects under `bronze/imdb/` and read a sample file.

In [ ]:
# Read first file and show sample records (self-contained: fetches list if needed)
import json
from src.utils.s3_client import get_s3_client
from src import config

client = get_s3_client()
bucket = config.S3_DATA_BUCKET
prefix = f"{config.BRONZE_PREFIX}imdb/"
resp = client.list_objects_v2(Bucket=bucket, Prefix=prefix)
objects = resp.get("Contents", [])

if objects:
    key = objects[0]["Key"]
    body = client.get_object(Bucket=bucket, Key=key)["Body"].read().decode("utf-8")
    lines = [l for l in body.strip().split("\n") if l]
    print(f"First 2 records from {key}:")
    for line in lines[:20]:
        rec = json.loads(line)
        print(f"  id={rec['id']}, label={rec['label']}, text_preview={rec['text'][:60]}...")
else:
    print("No objects found under", prefix)

In [ ]:
import json
from src.utils.s3_client import get_s3_client
from src import config

client = get_s3_client()
bucket = config.S3_DATA_BUCKET
prefix = f"{config.BRONZE_PREFIX}imdb/"

resp = client.list_objects_v2(Bucket=bucket, Prefix=prefix)
objects = resp.get("Contents", [])
print(f"Objects under {prefix}: {len(objects)}")
for obj in objects[:5]:
    print(f"  - {obj['Key']} ({obj['Size']} bytes)")

## 5. Run via CLI (alternative)

You can also use the project's CLI scripts from a terminal:

```bash
# Terminal 1: stream IMDb to Kafka
python -m src.ingestion.producer --mode batch --limit 100 --split train

# Terminal 2: consume to bronze (use --limit so it exits after ingesting)
python -m src.ingestion.bronze_consumer --batch-size 50 --limit 100 --split train
```